In [1]:
import torch
import mininn as mini
import torch.nn as nn

device = "cuda"
g = torch.Generator(device=device).manual_seed(42)

In [16]:
class Inception:
    def __init__(self, in_channels, *, device="cpu", generator=None):

        a = mini.Conv2D(in_channels=in_channels, out_channels=16, kernel_size=1, device=device, generator=g)
        a_act = mini.ReLU()
        
        self.branch_a = mini.Sequential([a,a_act])
        
        b_1 = mini.Conv2D(in_channels=in_channels, out_channels=16, kernel_size=1, device=device, generator=g)
        b1_act = mini.ReLU()
        
        b = mini.Conv2D(in_channels=16, out_channels=24, kernel_size=3, stride=1, padding=1, device=device, generator=g)
        b_act = mini.ReLU()
        
        self.branch_b = mini.Sequential([b_1, b1_act, b, b_act])
        
        c_1 = mini.Conv2D(in_channels=in_channels, out_channels=8, kernel_size=1, device=device, generator=g)
        c1_act = mini.ReLU()
        
        c = mini.Conv2D(in_channels=8, out_channels=12, kernel_size=5, stride=1, padding=2, device=device, generator=g)
        c_act = mini.ReLU()
        
        self.branch_c = mini.Sequential([c_1, c1_act, c, c_act])
        
        d = mini.MaxPool2D(3,1,1)
        d_1 = mini.Conv2D(in_channels=in_channels, out_channels=8, kernel_size=1, device=device, generator=g)
        d_act = mini.ReLU()
        
        self.branch_d = mini.Sequential([d, d_1, d_act])
    
    def __call__(self, x):
        
        output_a = self.branch_a(x)
        output_b = self.branch_b(x)
        output_c = self.branch_c(x)
        output_d = self.branch_d(x)
        
        output = torch.cat(
            (output_a, output_b, output_c, output_d),
            dim=1,
        )
        return output
    
    def parameters(self):

        return (
            self.branch_a.parameters()
            + self.branch_b.parameters()
            + self.branch_c.parameters()
            + self.branch_d.parameters()
        )
    
    def train(self):
        for branch in (
            self.branch_a, self.branch_b,
            self.branch_c, self.branch_d,
        ):
            branch.train()


    def eval(self):
        for branch in (
            self.branch_a, self.branch_b,
            self.branch_c, self.branch_d,
        ):
            branch.eval()


    def state_dict(self):
        state = {}

        for name in ("branch_a", "branch_b", "branch_c", "branch_d"):
            branch = getattr(self, name)

            for key, tensor in branch.state_dict().items():
                state[f"{name}.{key}"] = tensor

        return state


    def load_state_dict(self, state):
        expected = self.state_dict()

        # Validate every key and shape before changing parameters.
        if set(state) != set(expected):
            missing = set(expected) - set(state)
            unexpected = set(state) - set(expected)
            raise ValueError(
                f"Missing keys: {sorted(missing)}; "
                f"Unexpected keys: {sorted(unexpected)}"
            )

        for key, tensor in expected.items():
            if state[key].shape != tensor.shape:
                raise ValueError(f"Shape mismatch for {key}.")

        for name in ("branch_a", "branch_b", "branch_c", "branch_d"):
            prefix = f"{name}."
            branch_state = {
                key[len(prefix):]: tensor
                for key, tensor in state.items()
                if key.startswith(prefix)
            }
            getattr(self, name).load_state_dict(branch_state)


In [14]:
block = Inception()

# Match the convolution weights' device and dtype.
weight = block.branch_a.parameters()[0]
x = torch.randn(
    2, 64, 16, 16,
    device=weight.device,
    dtype=weight.dtype,
    requires_grad=True,
)

expected_shapes = {
    "a": (2, 16, 16, 16),
    "b": (2, 24, 16, 16),
    "c": (2, 12, 16, 16),
    "d": (2, 8, 16, 16),
}

with torch.no_grad():
    branches = [
        block.branch_a(x),
        block.branch_b(x),
        block.branch_c(x),
        block.branch_d(x),
    ]

    for (name, expected), result in zip(expected_shapes.items(), branches):
        print(f"Branch {name}: {tuple(result.shape)}")
        assert tuple(result.shape) == expected

    expected_output = torch.cat(branches, dim=1)

output = block(x)

assert tuple(output.shape) == (2, 60, 16, 16)
assert torch.isfinite(output).all()
assert torch.allclose(output, expected_output)

output.square().mean().backward()

assert x.grad is not None
assert torch.isfinite(x.grad).all()

for name in ("a", "b", "c", "d"):
    branch = getattr(block, f"branch_{name}")
    for parameter in branch.parameters():
        assert parameter.grad is not None
        assert torch.isfinite(parameter.grad).all()

print("Output shape:", tuple(output.shape))
print("Shape, concatenation, and gradient-flow checks passed.")
print(len(block.parameters()))

Branch a: (2, 16, 16, 16)
Branch b: (2, 24, 16, 16)
Branch c: (2, 12, 16, 16)
Branch d: (2, 8, 16, 16)
Output shape: (2, 60, 16, 16)
Shape, concatenation, and gradient-flow checks passed.
12


In [4]:
x = torch.tensor([[[[-4., -3.],
                    [-2., -1.]]]])

pool = mini.MaxPool2D(kernel_size=3, stride=1, padding=1)
output = pool(x)
expected = torch.full_like(x, -1.)

print(output)
print("Matches expected:", torch.allclose(output, expected))

tensor([[[[-1., -1.],
          [-1., -1.]]]])
Matches expected: True


In [10]:
block = Inception()
first_weight = block.branch_a.parameters()[0]
device, dtype = first_weight.device, first_weight.dtype

reference = nn.ModuleDict({
    "a": nn.Sequential(
        nn.Conv2d(64, 16, 1), nn.ReLU(),
    ),
    "b": nn.Sequential(
        nn.Conv2d(64, 16, 1), nn.ReLU(),
        nn.Conv2d(16, 24, 3, padding=1), nn.ReLU(),
    ),
    "c": nn.Sequential(
        nn.Conv2d(64, 8, 1), nn.ReLU(),
        nn.Conv2d(8, 12, 5, padding=2), nn.ReLU(),
    ),
    "d": nn.Sequential(
        nn.MaxPool2d(3, stride=1, padding=1),
        nn.Conv2d(64, 8, 1), nn.ReLU(),
    ),
}).to(device=device, dtype=dtype)

# Pair corresponding convolutions and copy parameters.
pairs = []

with torch.no_grad():
    for name in ("a", "b", "c", "d"):
        custom_branch = getattr(block, f"branch_{name}")

        custom_convs = [
            layer for layer in custom_branch.layers
            if isinstance(layer, mini.Conv2D)
        ]
        native_convs = [
            layer for layer in reference[name]
            if isinstance(layer, nn.Conv2d)
        ]

        assert len(custom_convs) == len(native_convs)

        for custom, native in zip(custom_convs, native_convs):
            native.weight.copy_(custom.weight)
            native.bias.copy_(custom.bias)
            custom.weight.grad = None
            custom.bias.grad = None
            pairs.append((custom, native))

x_custom = torch.randn(
    2, 64, 16, 16, device=device, dtype=dtype,
    requires_grad=True,
)
x_native = x_custom.detach().clone().requires_grad_(True)

y_custom = block(x_custom)
y_native = torch.cat(
    [reference[name](x_native) for name in ("a", "b", "c", "d")],
    dim=1,
)

upstream = torch.randn_like(y_custom)
y_custom.backward(upstream)
y_native.backward(upstream)

def compare(name, actual, expected):
    max_error = (actual - expected).abs().max().item()
    matches = torch.allclose(actual, expected, atol=1e-4, rtol=1e-3)
    print(f"{name}: matches={matches}, max_abs_error={max_error:.3e}")
    assert matches, f"{name} comparison failed"

compare("Output", y_custom, y_native)
compare("Input gradient", x_custom.grad, x_native.grad)

for i, (custom, native) in enumerate(pairs, start=1):
    compare(f"Conv {i} weight gradient", custom.weight.grad, native.weight.grad)
    compare(f"Conv {i} bias gradient", custom.bias.grad, native.bias.grad)

print("All native-reference comparisons passed.")

Output: matches=True, max_abs_error=2.861e-06
Input gradient: matches=True, max_abs_error=9.537e-07
Conv 1 weight gradient: matches=True, max_abs_error=7.629e-06
Conv 1 bias gradient: matches=True, max_abs_error=0.000e+00
Conv 2 weight gradient: matches=True, max_abs_error=1.144e-05
Conv 2 bias gradient: matches=True, max_abs_error=6.676e-06
Conv 3 weight gradient: matches=True, max_abs_error=1.907e-05
Conv 3 bias gradient: matches=True, max_abs_error=0.000e+00
Conv 4 weight gradient: matches=True, max_abs_error=1.526e-05
Conv 4 bias gradient: matches=True, max_abs_error=4.768e-06
Conv 5 weight gradient: matches=True, max_abs_error=1.240e-05
Conv 5 bias gradient: matches=True, max_abs_error=0.000e+00
Conv 6 weight gradient: matches=True, max_abs_error=2.289e-05
Conv 6 bias gradient: matches=True, max_abs_error=0.000e+00
All native-reference comparisons passed.


In [9]:
for name in ("a", "b", "c", "d"):
    xc = torch.randn(
        2, 64, 16, 16, device=device, dtype=dtype,
        requires_grad=True,
    )
    xn = xc.detach().clone().requires_grad_(True)

    yc = getattr(block, f"branch_{name}")(xc)
    yn = reference[name](xn)

    upstream = torch.randn_like(yc)
    yc.backward(upstream)
    yn.backward(upstream)

    print(f"\nBranch {name}")
    compare("Output", yc, yn)
    compare("Input gradient", xc.grad, xn.grad)


Branch a
Output: matches=True, max_abs_error=1.907e-06
Input gradient: matches=True, max_abs_error=3.576e-07

Branch b
Output: matches=True, max_abs_error=1.669e-06
Input gradient: matches=True, max_abs_error=4.768e-07

Branch c
Output: matches=True, max_abs_error=1.788e-06
Input gradient: matches=True, max_abs_error=3.874e-07

Branch d
Output: matches=True, max_abs_error=2.384e-06
Input gradient: matches=True, max_abs_error=7.153e-07


In [8]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

In [15]:
block = Inception()
restored = Inception()

assert len(block.parameters()) == 12
assert len(block.state_dict()) == 12

restored.load_state_dict(block.state_dict())

weight = block.parameters()[0]
x = torch.randn(
    2, 64, 16, 16,
    device=weight.device,
    dtype=weight.dtype,
)

block.eval()
restored.eval()

with torch.no_grad():
    original_output = block(x)
    restored_output = restored(x)

print("Parameter tensors:", len(block.parameters()))
print("State entries:", len(block.state_dict()))
print("Restored output matches:",
      torch.allclose(original_output, restored_output))

assert torch.allclose(original_output, restored_output)

Parameter tensors: 12
State entries: 12
Restored output matches: True


In [21]:
first = Inception(64, device=device, generator=g)
second = Inception(60, device=device, generator=g)

In [22]:
x = torch.randn(2, 64, 16, 16, device=device)

with torch.no_grad():
    h = first(x)
    y = second(h)

print(h.shape)  # (2, 60, 16, 16)
print(y.shape)  # (2, 60, 16, 16)

torch.Size([2, 60, 16, 16])
torch.Size([2, 60, 16, 16])


In [3]:
block = mini.Inception(
    in_channels=64,
    ch1=32,
    ch3_reduce=32, ch3=48,
    ch5_reduce=8, ch5=16,
    pool_proj=32,
    device=device,
    generator=g,
)

weight = block.parameters()[0]
x = torch.randn(
    2, 64, 32, 32,
    device=weight.device,
    dtype=weight.dtype,
    requires_grad=True,
)

y = block(x)
assert y.shape == (2, 128, 32, 32)
assert block.out_channels == 128
assert torch.isfinite(y).all()

y.square().mean().backward()

assert x.grad is not None
assert torch.isfinite(x.grad).all()

for p in block.parameters():
    assert p.grad is not None
    assert torch.isfinite(p.grad).all()

print("Output:", y.shape)
print("Configurable block gradient checks passed.")

Output: torch.Size([2, 128, 32, 32])
Configurable block gradient checks passed.
